# 02 - Checkpoint sweep - Victoria

**Your assignment:** LMEnt-1B-1E (epoch-count control) plus template-robustness runs

1B-1E sees the same data once instead of six times, which separates repetition from volume. The alt-template runs show the results are not a wording artifact.

## Before you run
1. **Settings -> Accelerator -> `GPU T4 x2`** (NOT P100 - it is sm_60 and crashes)
2. **Settings -> Internet -> On**
3. **Add Data -> search `lment-probes`** (Tomer's dataset) so everyone scores the identical
   probe set. If you skip this the notebook regenerates it and prints a hash - that hash must
   match Tomer's or the results cannot be pooled.
4. **Run All**, then send Tomer every `res__*.parquet` and `manifest__victoria.json`.

The sweep saves after every checkpoint and skips what is already done, so if the session dies
just Run All again.

## 1. Environment

In [ ]:
import sys, os, subprocess, json, time, gc, shutil, platform, hashlib, re

ON_KAGGLE = os.path.isdir("/kaggle")

def _writable(d):
    try:
        os.makedirs(d, exist_ok=True)
        t = os.path.join(d, ".wtest")
        with open(t, "w") as f: f.write("x")
        os.remove(t); return d
    except Exception:
        return None

SCRATCH = None
for cand in (["/kaggle/temp", "/tmp"] if ON_KAGGLE else ["./_scratch"]):
    SCRATCH = _writable(cand)
    if SCRATCH: break
assert SCRATCH, "no writable scratch directory"
OUT = _writable("/kaggle/working" if ON_KAGGLE else "./_out")
TMP = os.path.join(SCRATCH, "ckpt"); os.makedirs(TMP, exist_ok=True)
os.environ["HF_HOME"] = os.path.join(SCRATCH, "hf")     # must precede any HF import
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import transformers
    from packaging.version import parse as V
    if V(transformers.__version__) < V("4.47"):
        pipq("-U", "transformers>=4.47")
        os.execv(sys.executable, [sys.executable] + sys.argv)
except ImportError:
    pipq("-U", "transformers>=4.47")
import numpy as np, pandas as pd, transformers
try:
    import pyarrow
except ImportError:
    pipq("pyarrow"); import pyarrow

import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise SystemExit("Internet is OFF. Kaggle -> Settings -> Internet -> On. " + str(e))
print("scratch:", TMP, "| out:", OUT, "| transformers:", transformers.__version__)

## 2. Config and assignment

In [ ]:
PERSON = "victoria"

SEED         = 0
N_PER_BUCKET = 300          # 300 x 10 buckets = 3000 facts
N_CAND       = 10           # gold + 9 distractors -> chance = 0.100
GEN_TOKENS   = 10

BUCKET_EDGES  = [-1, 0, 1, 2, 4, 7, 13, 25, 60, 200, 10**9]
BUCKET_LABELS = ["0","1","2","3-4","5-7","8-13","14-25","26-60","61-200","200+"]

# Deterministic, hand-written. No LLM in the loop, so probe construction is reproducible.
TEMPLATES = {
    "director":       "The director of {s} is",
    "screenwriter":   "The screenwriter of {s} is",
    "genre":          "The genre of {s} is",
    "producer":       "The producer of {s} is",
    "author":         "The author of {s} is",
    "composer":       "The composer of {s} is",
    "country":        "{s} is located in the country of",
    "capital":        "The capital of {s} is",
    "place of birth": "{s} was born in the city of",
    "father":         "The father of {s} is",
    "sport":          "{s} plays the sport of",
    "occupation":     "The occupation of {s} is",
    "capital of":     "{s} is the capital of",
    "religion":       "The religion of {s} is",
    "mother":         "The mother of {s} is",
    "color":          "The color of {s} is",
}
STEPS_PER_EPOCH = 109672        # LMEnt: 1 epoch = 109,672 optimiser steps
TOKENIZER_REF   = ("dhgottesman/LMEnt-1B-6E", "step10000")   # identical across all LMEnt models

# Paraphrase sets, used only for the template-robustness check.
TEMPLATES_ALT1 = {
    "director":"{s} was directed by", "screenwriter":"{s} was written for the screen by",
    "genre":"{s} is a work in the genre of", "producer":"{s} was produced by",
    "author":"{s} was written by", "composer":"{s} was composed by",
    "country":"{s} can be found in the country of", "capital":"The capital city of {s} is",
    "place of birth":"{s} was born in", "father":"{s} has a father named",
    "sport":"{s} competes in the sport of", "occupation":"{s} works as a",
    "capital of":"{s} serves as the capital of", "religion":"{s} practices the religion of",
    "mother":"{s} has a mother named", "color":"{s} has the color",
}
TEMPLATES_ALT2 = {k: "{s}'s " + v for k, v in {
    "director":"director is", "screenwriter":"screenwriter is", "genre":"genre is",
    "producer":"producer is", "author":"author is", "composer":"composer is",
    "country":"country is", "capital":"capital is", "place of birth":"birthplace is",
    "father":"father is", "sport":"sport is", "occupation":"occupation is",
    "capital of":"capital region is", "religion":"religion is", "mother":"mother is",
    "color":"color is"}.items()}
TEMPLATE_SETS = {"main": TEMPLATES, "alt1": TEMPLATES_ALT1, "alt2": TEMPLATES_ALT2}

DENSE_EARLY = [0,1000,2000,3000,4000,5000,6000,7000,8000,9000,10000]
LATE_1B     = [20000,30000,40000,60000,80000,100000,130000,170000,
               220000,280000,350000,430000,520000,658032]
MATCHED     = [0,10000,20000,40000,60000,100000,150000,220000,300000,400000,500000,658032]
ONE_EPOCH   = [0,10000,20000,30000,40000,50000,60000,70000,80000,90000,100000,109672]
M1B, M600, M170, M1B1E = ("dhgottesman/LMEnt-1B-6E", "dhgottesman/LMEnt-600M-6E",
                          "dhgottesman/LMEnt-170M-6E", "dhgottesman/LMEnt-1B-1E")

ASSIGNMENTS = {                      # (model, steps, template_set)
  "tomer":    [(M1B,   DENSE_EARLY,   "main")],
  "chen":     [(M1B,   LATE_1B[:7],   "main"), (M170, MATCHED, "main")],
  "ofek":     [(M1B,   LATE_1B[7:],   "main"), (M600, MATCHED, "main")],
  "victoria": [(M1B1E, ONE_EPOCH,     "main"),
               (M1B,   [10000, 100000, 658032], "alt1"),
               (M1B,   [10000, 100000, 658032], "alt2")],
}
PLAN = ASSIGNMENTS[PERSON]
print("runner:", PERSON)
for m, st, ts in PLAN:
    print(f"  {m.split('/')[-1]:<18} {len(st):>2} checkpoints  templates={ts}")
print("total runs:", sum(len(st) for _, st, _ in PLAN))

## 3. Probe set

In [ ]:
def as_list(x):
    if x is None: return []
    return [str(v) for v in list(x)]

def build_probes(facts, tok, n_per_bucket=N_PER_BUCKET, seed=SEED):
    """Deterministic probe construction. Same seed -> byte-identical probe set."""
    def ntok(s): return len(tok(" " + s, add_special_tokens=False)["input_ids"])
    rng = np.random.default_rng(seed)
    sample = pd.concat([g.sample(min(n_per_bucket, len(g)), random_state=seed)
                        for _, g in facts.groupby("bucket", observed=True)]).reset_index(drop=True)
    pools    = {p: sorted(set(g.obj)) for p, g in facts.groupby("prop")}
    pool_len = {p: np.array([ntok(o) for o in pools[p]]) for p in pools}
    obj_freq = facts.groupby("obj").answer_num_chunks.max().to_dict()

    probes = []
    for r in sample.itertuples():
        pool, plens = pools[r.prop], pool_len[r.prop]
        gold_len = ntok(r.obj)
        banned   = {r.obj.lower()} | {a.lower() for a in as_list(r.possible_answers)}
        cands = [r.obj]
        for tol in (0, 1, 2, 99):                 # widen until we have enough
            idx = np.flatnonzero(np.abs(plens - gold_len) <= tol)
            rng.shuffle(idx)
            for j in idx:
                c = pool[j]
                if c.lower() in banned or c in cands: continue
                cands.append(c)
                if len(cands) == N_CAND: break
            if len(cands) == N_CAND: break
        if len(cands) < N_CAND: continue
        # Corpus prior among the DISTRACTORS only. Defining it over all candidates makes
        # chose_prior structurally impossible whenever the prior happens to be the gold,
        # which silently biases high-exposure buckets downward.
        dist_freq = [obj_freq.get(c, 0) for c in cands[1:]]
        probes.append({
            "fact_id": int(r.id), "subj": r.subj, "prop": r.prop, "obj": r.obj,
            "bucket": str(r.bucket), "n_shared": int(r.num_shared_chunks),
            "n_subject": int(r.subject_num_chunks), "subj_id": int(r.subj_id),
            "prompt": TEMPLATES[r.prop].format(s=r.subj),
            "candidates": cands,                       # index 0 is ALWAYS the gold
            "corpus_prior_idx": int(np.argmax(dist_freq)) + 1,   # never 0
            "possible_answers": as_list(r.possible_answers),
        })
    return probes

def probe_hash(probes):
    """Stable fingerprint. All four runners must report the same value."""
    payload = json.dumps([[p["fact_id"], p["candidates"]] for p in probes],
                         sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()[:16]

In [ ]:
from huggingface_hub import HfFileSystem
import pyarrow.parquet as pq

SCALAR_COLS = ["id","subj","prop","obj","subj_id","obj_id","s_pop","o_pop",
               "question","possible_answers",
               "subject_num_chunks","answer_num_chunks","num_shared_chunks"]

def load_popqa_kas(columns):
    """Column-selective remote parquet read: never downloads the huge list columns
       unless we ask for them."""
    fs = HfFileSystem(); parts=[]
    for i in range(9):
        p = f"datasets/dhgottesman/popqa-kas/data/train-0000{i}-of-00009.parquet"
        with fs.open(p, "rb") as fh:
            parts.append(pq.ParquetFile(fh).read(columns=columns).to_pandas())
        print(f"  shard {i}", flush=True)
    return pd.concat(parts, ignore_index=True)

In [ ]:
import glob
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(TOKENIZER_REF[0], subfolder=TOKENIZER_REF[1], use_fast=True)

hits = (sorted(glob.glob("/kaggle/input/**/probes.json", recursive=True))
        + sorted(glob.glob("./**/probes.json", recursive=True)))
if hits:
    payload = json.load(open(hits[0]))
    probes, PROBE_HASH = payload["probes"], payload["probe_hash"]
    assert probe_hash(probes) == PROBE_HASH, "probes.json is corrupt (hash mismatch)"
    print("loaded shared probe set from", hits[0])
else:
    print("No probes.json found - regenerating deterministically from the same seed.")
    print("Attach Tomer's 'lment-probes' Kaggle Dataset if the hash below does not match.")
    facts = load_popqa_kas(SCALAR_COLS)
    facts["bucket"] = pd.cut(facts.num_shared_chunks, BUCKET_EDGES, labels=BUCKET_LABELS)
    probes = build_probes(facts, tok)
    PROBE_HASH = probe_hash(probes)

print("PROBE_HASH:", PROBE_HASH)
print(len(probes), "probes")
assert len(probes) > 0
UNIQ_CAND = sorted({c for p in probes for c in p["candidates"]})
CIDX = {c: i for i, c in enumerate(UNIQ_CAND)}
NEUTRAL = [("The answer is", c) for c in UNIQ_CAND]
print(len(UNIQ_CAND), "unique candidates (neutral scores are deduped across facts)")

## 4. Model runtime

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

def preflight_gpu():
    name  = torch.cuda.get_device_name(0)
    cap   = torch.cuda.get_device_capability(0)
    sm    = "sm_" + str(cap[0]) + str(cap[1])
    archs = list(torch.cuda.get_arch_list())
    print("GPU:", name, "| capability:", sm)
    if sm not in archs:
        print("this torch was built for:", ", ".join(archs))
        print("STOP - no kernels for this GPU. Kaggle P100 (sm_60) is broken.")
        print("FIX: Settings -> Accelerator -> 'GPU T4 x2', then Run All.")
        raise SystemExit("incompatible GPU " + sm)
    emb = torch.nn.Embedding(16, 8).cuda()
    emb(torch.zeros(2, dtype=torch.long, device="cuda")).float().sum().item()
    print("GPU preflight: OK")
    return sm

def fetch_ckpt(model_id, sub):
    d = os.path.join(TMP, model_id.split("/")[-1] + "__" + sub)
    snapshot_download(model_id, allow_patterns=[sub + "/*"], local_dir=d)
    return d

def load_from(d, sub, dt):
    try:    m = AutoModelForCausalLM.from_pretrained(os.path.join(d, sub), dtype=dt)
    except TypeError:
            m = AutoModelForCausalLM.from_pretrained(os.path.join(d, sub), torch_dtype=dt)
    return m.to("cuda").eval()

def free_model(m):
    del m; gc.collect(); torch.cuda.empty_cache()

def all_finite(scored):
    v = np.array([x[0] for x in scored], dtype=float)
    return bool(np.isfinite(v).all()), int((~np.isfinite(v)).sum()), len(v)

def _encode(tok, pairs):
    seqs, alen = [], []
    for p, a in pairs:
        pid = tok(p, add_special_tokens=False)["input_ids"]
        aid = tok(" " + a, add_special_tokens=False)["input_ids"]
        seqs.append([tok.bos_token_id] + pid + aid); alen.append(len(aid))
    return seqs, alen

@torch.no_grad()
def score_pairs(model, tok, pairs, batch=96):
    """Sum log P(answer | prompt). Uses logsumexp on shifted logits so we never
       materialise a second (B, L, V) tensor - that is what caps the batch size."""
    out = []
    i = 0
    while i < len(pairs):
        chunk = pairs[i:i+batch]
        try:
            seqs, alen = _encode(tok, chunk)
            L = max(len(s) for s in seqs)
            inp = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
            att = torch.zeros((len(seqs), L), dtype=torch.long)
            msk = torch.zeros((len(seqs), L-1), dtype=torch.float)
            for j, s in enumerate(seqs):
                inp[j,:len(s)] = torch.tensor(s); att[j,:len(s)] = 1
                e, n = len(s), alen[j]
                msk[j, e-n-1:e-1] = 1.0        # lp index k holds logprob of token k+1
            lg  = model(input_ids=inp.cuda(), attention_mask=att.cuda()).logits.float()
            cur = lg[:, :-1, :]
            nxt = inp[:, 1:].cuda()
            lp  = cur.gather(-1, nxt[:,:,None]).squeeze(-1) - torch.logsumexp(cur, dim=-1)
            tot = (lp * msk.cuda()).sum(1)
            for j in range(len(seqs)): out.append((tot[j].item(), alen[j]))
            i += batch
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if batch == 1: raise
            batch = max(1, batch // 2)
            print("  OOM -> batch", batch, flush=True)
    return out

@torch.no_grad()
def generate(model, tok, prompts, batch=48):
    texts, confs = [], []
    tok.padding_side = "left"
    for i in range(0, len(prompts), batch):
        enc = tok(prompts[i:i+batch], return_tensors="pt", padding=True,
                  add_special_tokens=True).to("cuda")
        o = model.generate(**enc, max_new_tokens=GEN_TOKENS, do_sample=False,
                           output_scores=True, return_dict_in_generate=True,
                           pad_token_id=tok.pad_token_id)
        new = o.sequences[:, enc["input_ids"].shape[1]:]
        st  = torch.stack(o.scores, 1).float()
        lp  = F.log_softmax(st, -1).gather(-1, new[:,:,None]).squeeze(-1)
        live = (new != tok.pad_token_id).float()
        mlp = (lp*live).sum(1) / live.sum(1).clamp(min=1)
        texts += tok.batch_decode(new, skip_special_tokens=True)
        confs += torch.exp(mlp).tolist()
    tok.padding_side = "right"
    return texts, confs

def norm_txt(s):
    return re.sub(r"[^a-z0-9 ]", "", (s or "").lower()).strip()

## 5. Run the sweep

In [ ]:
SM = preflight_gpu()

# The pilot established this empirically: LMEnt stores fp32 weights whose activations reach
# ~3.1e6, which is 47x past fp16's ceiling, so fp16 yields all-NaN logits. bf16 is correct but
# slower than fp32 on T4 (no native bf16 hardware). Verified again below, cheaply.
DTYPE, DTYPE_NAME = torch.float32, "float32"

TIMING, DONE = {}, []

def run_one(model_id, sub_step, tset):
    short = model_id.split("/")[-1]
    sub   = "step" + str(sub_step)
    tag   = f"{PERSON}__{short}__{tset}__{sub}"
    path  = os.path.join(OUT, "res__" + tag + ".parquet")
    if os.path.exists(path):
        print(tag, "cached"); return

    t0 = time.time()
    d  = fetch_ckpt(model_id, sub)
    t_dl = time.time() - t0
    model = load_from(d, sub, DTYPE)

    tpl     = TEMPLATE_SETS[tset]
    prompts = [tpl[p["prop"]].format(s=p["subj"]) for p in probes]
    pairs   = [(pr, c) for pr, p in zip(prompts, probes) for c in p["candidates"]]

    t1 = time.time()
    sc  = score_pairs(model, tok, pairs)
    nsc = score_pairs(model, tok, NEUTRAL)
    gtxt, gconf = generate(model, tok, prompts)
    t_gpu = time.time() - t1
    free_model(model); shutil.rmtree(d, ignore_errors=True)

    for label, arr in (("probe", sc), ("neutral", nsc)):
        ok, nbad, ntot = all_finite(arr)
        if not ok:
            raise RuntimeError(f"{tag}: {nbad}/{ntot} non-finite {label} scores under "
                               f"{DTYPE_NAME}. Numerical failure - do not trust anything downstream.")

    neutral_lp = {c: nsc[CIDX[c]][0] for c in UNIQ_CAND}
    rows = []
    for qi, p in enumerate(probes):
        blk = sc[qi*N_CAND:(qi+1)*N_CAND]
        s   = np.array([b[0] for b in blk]); nt = np.array([b[1] for b in blk])
        nlp = np.array([neutral_lp[c] for c in p["candidates"]])
        # the model's own marginal favourite among the DISTRACTORS
        model_prior_idx = int(np.argmax(nlp[1:])) + 1
        r = {"fact_id": p["fact_id"], "person": PERSON, "model": short, "templates": tset,
             "step": sub_step, "bucket": p["bucket"], "split": p.get("split", "na"),
             "n_shared": p["n_shared"], "n_subject": p["n_subject"],
             "corpus_prior_idx": p["corpus_prior_idx"], "model_prior_idx": model_prior_idx}
        for name, sco in (("norm", s/nt), ("sum", s), ("pmi", s-nlp)):
            pr  = np.exp(sco - sco.max()); pr /= pr.sum()
            top = int(np.argmax(pr))
            r["acc_"+name]   = int(top == 0)
            r["conf_"+name]  = float(pr.max())
            r["pgold_"+name] = float(pr[0])
            r["top_"+name]   = top
        wrong = r["top_norm"] != 0
        r["chose_corpus_prior"] = int(wrong and r["top_norm"] == p["corpus_prior_idx"])
        r["chose_model_prior"]  = int(wrong and r["top_norm"] == model_prior_idx)
        g = gtxt[qi]
        r["gen_text"]    = g
        r["gen_correct"] = int(any(norm_txt(a) and norm_txt(a) in norm_txt(g)
                                   for a in p["possible_answers"]))
        r["gen_conf"]    = float(gconf[qi])
        rows.append(r)

    df = pd.DataFrame(rows)
    num = [c for c in df.columns if df[c].dtype.kind == "f"]
    nbad = int(df[num].isna().sum().sum())
    if nbad:
        raise RuntimeError(f"{tag}: {nbad} NaN in results - refusing to save.")
    df.to_parquet(path, index=False)
    TIMING[tag] = {"dl_s": round(t_dl,1), "gpu_s": round(t_gpu,1),
                   "total_s": round(time.time()-t0,1)}
    DONE.append(tag)
    print(f"{tag}: {TIMING[tag]['total_s']:6.0f}s (dl {t_dl:.0f}s gpu {t_gpu:.0f}s) "
          f"acc={df.acc_norm.mean():.3f} conf={df.conf_norm.mean():.3f}", flush=True)

ALL = [(m, s, ts) for m, steps, ts in PLAN for s in steps]
print(len(ALL), "runs queued")
for k, (m, s, ts) in enumerate(ALL):
    print(f"--- {k+1}/{len(ALL)} ---", flush=True)
    run_one(m, s, ts)
print("")
print("SWEEP COMPLETE:", len(DONE), "new,", len(ALL)-len(DONE), "already cached")

## 6. Summary and hand-off

In [ ]:
files = sorted(glob.glob(os.path.join(OUT, "res__*.parquet")))
res = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
num = [c for c in res.columns if res[c].dtype.kind == "f"]
assert not res[num].isna().any().any(), "NaN in pooled results"

print(len(res), "rows |", res.model.nunique(), "models |", len(files), "files")
print(res.groupby(["model","templates"]).step.nunique().to_string())
print("")
piv = res[res.templates=="main"].groupby(["bucket","step"], observed=True)[["acc_norm","conf_norm"]].mean()
if len(piv):
    last = res[res.step == res.step.max()]
    print("at the last step:")
    print((last.groupby("bucket", observed=True)[["acc_norm","conf_norm"]].mean()
              .reindex(BUCKET_LABELS).round(3)).to_string())

manifest = {"person": PERSON, "probe_hash": PROBE_HASH, "dtype": DTYPE_NAME,
            "gpu": torch.cuda.get_device_name(0), "sm": SM,
            "torch": torch.__version__, "transformers": transformers.__version__,
            "n_probes": len(probes), "files": [os.path.basename(f) for f in files],
            "timing": TIMING, "rows": len(res),
            "generated": time.strftime("%Y-%m-%d %H:%M:%S")}
json.dump(manifest, open(os.path.join(OUT, f"manifest__{PERSON}.json"), "w"), indent=2)

print("")
print("=" * 72)
print("SEND BACK TO TOMER: every res__*.parquet plus manifest__" + PERSON + ".json")
print("=" * 72)
print("PROBE_HASH:", PROBE_HASH, " <-- must match the other three runners")
for f in sorted(os.listdir(OUT)):
    print(f"  {f}  ({os.path.getsize(os.path.join(OUT,f))/1e6:.2f} MB)")